In [1]:

import csv
import os
from abc import ABC, abstractmethod
from datetime import datetime


In [2]:
class Transaction(ABC):

    def __init__(self, amount, category, description, date):
        self.__amount = amount
        self.__category = category
        self.__description = description
        self.__date = date

    # Getters
    def get_amount(self):
        return self.__amount

    def get_category(self):
        return self.__category

    def get_description(self):
        return self.__description

    def get_date(self):
        return self.__date

    # Setters
    def set_amount(self, amount):
        if amount > 0:
            self.__amount = amount
        else:
            raise ValueError("Amount must be greater than zero.")

    def set_category(self, category):
        if category.strip():
            self.__category = category

    def set_description(self, description):
        self.__description = description

    def set_date(self, date):
        self.__date = date

    @abstractmethod
    def get_type(self):
        pass


# ==========================================
# CLASS 2: INCOME CLASS (INHERITANCE)
# ==========================================

class Income(Transaction):

    def get_type(self):
        return "Income"


# ==========================================
# CLASS 3: EXPENSE CLASS (INHERITANCE)
# ==========================================

class Expense(Transaction):

    def get_type(self):
        return "Expense"


# ==========================================
# CLASS 4: EXPENSE TRACKER
# ==========================================

class ExpenseTracker:

    def __init__(self):
        self.transactions = []
        self.filename = "transactions.csv"
        self.load_records()

    # Add a transaction
    def add_transaction(self, transaction):
        self.transactions.append(transaction)
        self.save_records()
        print("\nTransaction added successfully!")

    # Save records to CSV
    def save_records(self):
        try:
            with open(self.filename, "w", newline="",
                      encoding="utf-8") as file:

                writer = csv.writer(file)

                writer.writerow([
                    "Type", "Amount", "Category",
                    "Description", "Date"
                ])

                for transaction in self.transactions:
                    writer.writerow([
                        transaction.get_type(),
                        transaction.get_amount(),
                        transaction.get_category(),
                        transaction.get_description(),
                        transaction.get_date()
                    ])

        except OSError as error:
            print("Error saving records:", error)

    # Load previously saved records
    def load_records(self):
        if not os.path.exists(self.filename):
            return

        try:
            with open(self.filename, "r", newline="",
                      encoding="utf-8") as file:

                reader = csv.DictReader(file)

                for row in reader:
                    amount = float(row["Amount"])
                    category = row["Category"]
                    description = row["Description"]
                    date = row["Date"]

                    if row["Type"] == "Income":
                        transaction = Income(
                            amount, category, description, date
                        )

                    elif row["Type"] == "Expense":
                        transaction = Expense(
                            amount, category, description, date
                        )

                    else:
                        continue

                    self.transactions.append(transaction)

            print("Saved records loaded successfully.")

        except (OSError, ValueError, KeyError, csv.Error) as error:
            print("Error loading records:", error)

    # Display all transactions
    def show_transactions(self):
        if not self.transactions:
            print("\nNo transactions found.")
            return

        print("\n========== ALL TRANSACTIONS ==========")

        for transaction in self.transactions:
            print("--------------------------------------")
            print("Type:", transaction.get_type())
            print("Amount: Rs.", transaction.get_amount())
            print("Category:", transaction.get_category())
            print("Description:", transaction.get_description())
            print("Date:", transaction.get_date())

        print("--------------------------------------")

    # Calculate monthly summary
    def monthly_summary(self):
        month = input("Enter month (YYYY-MM): ").strip()

        try:
            datetime.strptime(month, "%Y-%m")
        except ValueError:
            print("Invalid month. Use YYYY-MM, e.g. 2026-10.")
            return

        total_income = 0
        total_expenses = 0

        for transaction in self.transactions:
            if transaction.get_date().startswith(month):

                if transaction.get_type() == "Income":
                    total_income += transaction.get_amount()

                elif transaction.get_type() == "Expense":
                    total_expenses += transaction.get_amount()

        savings = total_income - total_expenses

        print("\n========== MONTHLY SUMMARY ==========")
        print("Month:", month)
        print("Total Income: Rs.", round(total_income, 2))
        print("Total Expenses: Rs.", round(total_expenses, 2))
        print("Savings: Rs.", round(savings, 2))

    # Search transactions
    def search_records(self):
        keyword = input(
            "Enter category or description to search: "
        ).strip().lower()

        found = False

        for transaction in self.transactions:
            if (keyword in transaction.get_category().lower()
                    or keyword in transaction.get_description().lower()):

                print("\nType:", transaction.get_type())
                print("Amount: Rs.", transaction.get_amount())
                print("Category:", transaction.get_category())
                print("Description:", transaction.get_description())
                print("Date:", transaction.get_date())

                found = True

        if not found:
            print("No matching records found.")


# ==========================================
# FUNCTION: GET TRANSACTION DETAILS
# ==========================================

def get_transaction_details():

    while True:
        try:
            amount = float(input("Enter amount (Rs.): "))

            if amount <= 0:
                print("Amount must be greater than zero.")
                continue

            break

        except ValueError:
            print("Please enter a valid number.")

    category = input("Enter category: ").strip()

    while not category:
        print("Category cannot be empty.")
        category = input("Enter category: ").strip()

    description = input("Enter description: ").strip()

    while True:
        date = input(
            "Enter date (YYYY-MM-DD), or press Enter for today: "
        ).strip()

        if date == "":
            date = datetime.now().strftime("%Y-%m-%d")
            break

        try:
            datetime.strptime(date, "%Y-%m-%d")
            break

        except ValueError:
            print("Invalid date. Example: 2026-10-26.")

    return amount, category, description, date


# ==========================================
# FUNCTION: MAIN MENU
# ==========================================

def main():

    tracker = ExpenseTracker()

    while True:
        print("\n================================")
        print("       EXPENSE TRACKER")
        print("================================")
        print("1. Record Income")
        print("2. Record Expense")
        print("3. View All Transactions")
        print("4. Monthly Summary")
        print("5. Search Transactions")
        print("6. Exit")
        print("================================")

        choice = input("Enter your choice (1-6): ").strip()

        try:

            if choice == "1":
                amount, category, description, date = (
                    get_transaction_details()
                )

                transaction = Income(
                    amount, category, description, date
                )

                tracker.add_transaction(transaction)

            elif choice == "2":
                amount, category, description, date = (
                    get_transaction_details()
                )

                transaction = Expense(
                    amount, category, description, date
                )

                tracker.add_transaction(transaction)

            elif choice == "3":
                tracker.show_transactions()

            elif choice == "4":
                tracker.monthly_summary()

            elif choice == "5":
                tracker.search_records()

            elif choice == "6":
                print("\nThank you for using Expense Tracker!")
                break

            else:
                print("Invalid choice. Please enter 1 to 6.")

        except (ValueError, OSError) as error:
            print("Something went wrong:", error)


# ==========================================
# START THE PROJECT
# ==========================================

if __name__ == "__main__":
    main()



       EXPENSE TRACKER
1. Record Income
2. Record Expense
3. View All Transactions
4. Monthly Summary
5. Search Transactions
6. Exit


Enter your choice (1-6):  1
Enter amount (Rs.):  50000
Enter category:  Air Conditioner EMI
Enter description:  Air conditioner emi
Enter date (YYYY-MM-DD), or press Enter for today:  22-10-2024


Invalid date. Example: 2026-10-26.


Enter date (YYYY-MM-DD), or press Enter for today:  



Transaction added successfully!

       EXPENSE TRACKER
1. Record Income
2. Record Expense
3. View All Transactions
4. Monthly Summary
5. Search Transactions
6. Exit


...

Invalid month. Use YYYY-MM, e.g. 2026-10.

       EXPENSE TRACKER
1. Record Income
2. Record Expense
3. View All Transactions
4. Monthly Summary
5. Search Transactions
6. Exit


Enter your choice (1-6):  5
Enter category or description to search:  Miscellaneous



Type: Expense
Amount: Rs. 6500.0
Category: Miscellaneous
Description: Miscellaneous
Date: 2026-10-26

       EXPENSE TRACKER
1. Record Income
2. Record Expense
3. View All Transactions
4. Monthly Summary
5. Search Transactions
6. Exit


Enter your choice (1-6):  6



Thank you for using Expense Tracker!
